# How much extra copper could electric cars and data centres need in 2030, next to today's mine output?

**Business question (whole project):** who needs copper, who supplies it, and what does the price do when they collide? This notebook covers the third hypothesis: **demand from EVs and data centres is a scenario, not a fact.**

**What it is.** Scenario arithmetic: published scenario volumes times copper-per-unit figures, set against what the world mines today. It is **not a forecast**, not a statement about what will happen, and not investment advice. Every input has a source and a reliability rating (section 1); the ranges are source ranges, not confidence intervals.

**Main year 2030, with 2035 beside it.** The IEA EV file has actual sales for 2025 and projections for **2035 only**, so 2030 is interpolated (two ways). 2035 is IEA's own scenario year, shown next to it so the interpolation does not carry the result alone.

**The measure.** Copper per year in the target year, as a percentage of **USGS world mine production for 2025 (an estimate)**. The wording is always "equivalent to X% of today's mine output". There is no supply projection in this project, so this is not a comparison of demand with future supply.

- **EV transition (headline EV term)** = extra EVs x (copper in an EV - copper in the petrol/diesel car it replaces).
- **Car-market growth (separate, not an EV effect)** = change in total car sales x copper in a petrol/diesel car. Total sales are *derived* as EV sales divided by the EV share, from rounded IEA numbers (plus interpolation for 2030). They are not EV data.
- **Data centres** use installed **capacity** (GW), not electricity. **Main figure: the copper in the capacity built in the target year** (gross, always positive). **Secondary figure: the extra over the 2024 build**, which subtracts the copper in 2024's build and can be negative when the build slows. Total capacity is the main basis; **IT capacity is shown as a prominent second result**.
- **The two effects together, on their own baselines** = the EV transition term (against 2025) plus the data-centre extra over the 2024 build. The two are shown as separate bars first and the total is labelled that way, because the two baselines differ (2025 for cars, 2024 for data centres).

**The IEA scenarios.** The IEA's Global EV Outlook 2026 calls the Current Policies Scenario (CPS) and Stated Policies Scenario (STEPS) "exploratory scenarios". Its text says CO2 and fuel-economy standards are the main driver of rising EV sales outside China in the CPS, and that STEPS follows a similar trajectory with higher growth, especially outside Europe and China (source S21). The pages read give no detailed list of the policies in each. In the data, CPS has fewer EV sales than STEPS in 2035 (49 and 55 million cars).

**The reference case** (CPS, straight-line 2030 path, Base Case data centres, even build, total capacity, middle copper figures) is a **convenience choice, not the most likely case**. The sensitivity section shows what happens when each assumption moves.

Run `python copper_database/build_db.py --raw .` first so that `copper.db` exists.

In [ ]:
import sqlite3
import textwrap
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# Four colours from the validated reference palette, in fixed order (blue, orange, aqua, yellow)
BLUE, ORANGE, AQUA, YELLOW, INK, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#0b0b0b", "#dcdcd8"
plt.rcParams.update({
    "figure.figsize": (9, 4.4), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8984", "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10,
    "text.color": INK, "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "lines.linewidth": 1.8, "legend.frameon": False,
})

con = sqlite3.connect("../copper_database/copper.db")
rd = lambda sql: pd.read_sql(sql, con)
assump = rd("SELECT * FROM mart_demand_assumptions")
issues = rd("SELECT issue_id, status, source_id, description FROM known_issues")
path = rd("SELECT * FROM mart_ev_sales_path")
cap = rd("SELECT * FROM stg_datacentre_capacity_scenarios")
dcp = rd("SELECT * FROM mart_dc_capacity_path")
dcc = rd("SELECT * FROM mart_dc_copper_cases")
head = rd("SELECT * FROM mart_demand_headline")
sens = rd("SELECT * FROM mart_demand_sensitivity")
ctx = rd("SELECT * FROM mart_demand_context").set_index("metric")
MINE = ctx.loc["world_mine_production_2025e_t", "value"]
print(f"Base for every percentage: USGS world mine production 2025 estimate, {MINE / 1e6:.1f} million tonnes")

kt = lambda s: (s / 1000).round(0)

## 1. Every assumption, its source and its reliability
*Kind* says whether a row is data, a range from a source (the middle is only the middle of the range, not a best estimate) or a project assumption. *Reliability* is the rating in the sources register: **high** = primary source; **medium** and **low** = secondary articles; **assumption** = my own modelling choice, not data.

In [ ]:
a = assump.copy()
a["low"], a["mid"], a["high"] = a["low"].round(2), a["mid"].round(2), a["high"].round(2)
a[["name", "unit", "low", "mid", "high", "source_id", "reliability", "kind"]].set_index("name")

**Known issues, recorded in the data checks.** Two are unresolved and four are limitations of the data.

In [ ]:
issues.set_index("issue_id")

## 2. Electric cars

### 2.1 How many electric cars could be sold by 2030 and 2035?
Solid dots are IEA figures (2025 actual, 2035 scenario). The 2030 points are **interpolated**: a circle for the straight line between the two, a diamond for constant yearly growth. The IEA publishes no 2030 value in this file.

In [ ]:
fig, ax = plt.subplots()
cols = {"CPS": BLUE, "STEPS": ORANGE}
for sc, c in cols.items():
    d = path[path.iea_scenario == sc].set_index(["year", "path_method"])
    y25, y35 = d.loc[(2025, "direct"), "ev"] / 1e6, d.loc[(2035, "direct"), "ev"] / 1e6
    lin, cg = d.loc[(2030, "linear"), "ev"] / 1e6, d.loc[(2030, "constant_growth"), "ev"] / 1e6
    ax.plot([2025, 2030, 2035], [y25, lin, y35], color=c, ls=":", lw=1.4)
    ax.plot([2025, 2030, 2035], [y25, cg, y35], color=c, ls="--", lw=1.2)
    ax.scatter([2025, 2035], [y25, y35], color=c, s=42, zorder=3)
    ax.scatter([2030], [lin], facecolor="white", edgecolor=c, s=48, linewidth=1.8, zorder=3, marker="o")
    ax.scatter([2030], [cg], facecolor="white", edgecolor=c, s=48, linewidth=1.8, zorder=3, marker="D")
    ax.annotate(f"{sc}: {y35:.0f} million", (2035, y35), xytext=(8, 0), textcoords="offset points", va="center", fontsize=9)
ax.annotate("2025: 21 million", (2025, 21), xytext=(10, -14), textcoords="offset points", ha="left", fontsize=9)
ax.set_xticks([2025, 2030, 2035])
ax.set_xlim(2024.5, 2035)
ax.figure.subplots_adjust(right=0.82)
ax.set_ylabel("World electric car sales (million a year)\nBEV + PHEV")
ax.set_title("How many electric cars could be sold by 2030 and 2035?")
h = [Line2D([], [], marker="o", color=BLUE, lw=0, label="IEA CPS"), Line2D([], [], marker="o", color=ORANGE, lw=0, label="IEA STEPS"),
     Line2D([], [], marker="o", color="#52514e", mfc="white", mew=1.6, lw=0, label="2030 interpolated, straight line"),
     Line2D([], [], marker="D", color="#52514e", mfc="white", mew=1.6, lw=0, label="2030 interpolated, constant growth")]
ax.legend(handles=h, loc="upper left", fontsize=8)
plt.show()

### 2.2 What the IEA numbers imply for total car sales
Total sales are **derived** (EV sales / EV share), and the IEA rounds EV sales to two significant figures and shares to whole percent. The band shows the range that rounding allows. This derived total is what the car-market growth term uses; it is not EV data.

In [ ]:
p = path.copy()
p["BEV (million)"] = (p["bev"] / 1e6).round(1)
p["PHEV (million)"] = (p["phev"] / 1e6).round(1)
p["EV share (%)"] = p["share_pct"].round(1)
p["total car sales, derived (million)"] = (p["total_sales_central"] / 1e6).round(1)
p["rounding band (million)"] = (p["total_sales_min"] / 1e6).round(1).astype(str) + " to " + (p["total_sales_max"] / 1e6).round(1).astype(str)
p["non-EV sales, derived (million)"] = (p["non_ev_sales_central"] / 1e6).round(1)
p[~((p.year == 2025) & (p.iea_scenario == "STEPS"))].set_index(["iea_scenario", "year", "path_method"])[
    ["BEV (million)", "PHEV (million)", "EV share (%)", "total car sales, derived (million)", "rounding band (million)", "non-EV sales, derived (million)"]]

### 2.3 The two EV-related terms, kept apart
Thousand tonnes of copper a year compared with 2025, with the low to high range from the source copper figures (BEV 82-91 kg, PHEV 59-68 kg, petrol/diesel mid-size car 18-22 kg). The **car-market growth** term is not an EV effect and is **not** added to the EV transition term. *Interpolated* = 2030 values between the IEA's 2025 and 2035 numbers.

In [ ]:
def comp_table(component_id, years):
    d = head[(head.component_id == component_id) & (head.year.isin(years))].copy()
    out = pd.DataFrame({
        "year": d["year"], "IEA scenario": d["variant"], "2030 path": d["path_method"].replace({"direct": "IEA figure"}),
        "low (kt)": kt(d["low_t"]), "mid (kt)": kt(d["mid_t"]), "high (kt)": kt(d["high_t"]),
        "mid (% of 2025e mine output)": d["mid_pct_of_mine"].round(1),
        "range (% of mine output)": d["low_pct_of_mine"].round(1).astype(str) + " to " + d["high_pct_of_mine"].round(1).astype(str),
    })
    return out.sort_values(["year", "IEA scenario", "2030 path"]).set_index(["year", "IEA scenario", "2030 path"])


print("EV transition: extra EVs x (EV copper - petrol-car copper)")
comp_table("ev_transition", [2030, 2035])

In [ ]:
print("Car-market growth: change in total car sales x petrol-car copper (derived from rounded shares; band = IEA rounding)")
comp_table("car_market_growth", [2030, 2035])

**Hybrids.** The IEA EV data counts only BEV, PHEV and fuel-cell cars and has no hybrid (HEV) split, so hybrids sit in the non-EV group, which is treated as an 18-22 kg petrol/diesel car. Hybrids carry about 39-45 kg (source S11). The data cannot split them out, so no data-based sensitivity is possible. The **bounding case** below treats *every* non-EV car as a hybrid (42 kg) - an extreme that the data do not support, shown only to bound the effect.

In [ ]:
s_ = sens[sens.bar_order.isin([0, 7])][["alt_label", "ev_transition_t", "market_growth_t", "dc_extra_vs_base_t", "headline_total_t", "headline_total_pct_of_mine"]].copy()
for c in ["ev_transition_t", "market_growth_t", "dc_extra_vs_base_t", "headline_total_t"]:
    s_[c] = kt(s_[c])
s_.columns = ["case", "EV transition (kt)", "car-market growth (kt)", "data-centre extra (kt)", "headline total (kt)", "headline total (% of mine output)"]
s_["headline total (% of mine output)"] = s_["headline total (% of mine output)"].round(2)
s_.set_index("case")

## 3. Data centres

### 3.1 How much data-centre capacity could be built by 2030 and 2035?
Installed capacity (GW) in the four IEA cases. 2024 is an IEA estimate; 2035 values (hollow dots, dashed) are marked **exploratory** by the IEA.

In [ ]:
fig, ax = plt.subplots()
cases = {"Lift-Off": BLUE, "Base": ORANGE, "High Efficiency": AQUA, "Headwinds": YELLOW}
tot = cap[cap.basis == "Whole data centre"]
y24 = tot[(tot.iea_case == "Estimate") & (tot.year == 2024)]["capacity_gw"].iloc[0]
for case, c in cases.items():
    d = tot[tot.iea_case == case].sort_values("year")
    c30, c35 = d[d.year == 2030]["capacity_gw"].iloc[0], d[d.year == 2035]["capacity_gw"].iloc[0]
    ax.plot([2024, 2030], [y24, c30], color=c)
    ax.plot([2030, 2035], [c30, c35], color=c, ls="--")
    ax.scatter([2030], [c30], color=c, s=40, zorder=3)
    ax.scatter([2035], [c35], facecolor="white", edgecolor=c, s=44, linewidth=1.7, zorder=3)
    ax.annotate(f"{case}: {d[d.year == 2035]['capacity_gw'].iloc[0]:.0f} GW", (2035, d[d.year == 2035]["capacity_gw"].iloc[0]), xytext=(8, 0), textcoords="offset points", va="center", fontsize=9)
ax.annotate(f"2024: {y24:.0f} GW", (2024, y24), xytext=(10, -14), textcoords="offset points", ha="left", fontsize=9)
ax.set_xticks([2024, 2030, 2035])
ax.set_xlim(2023.7, 2035)
ax.figure.subplots_adjust(right=0.78)
ax.set_ylabel("World installed data-centre capacity (GW), total")
ax.set_title("How much data-centre capacity could be built by 2030 and 2035?")
plt.show()

### 3.2 Build per year under two paths
The IEA gives capacity for 2024 and 2030 only, not the yearly build. **Even path:** the same GW every year from 2025. **Constant-growth path:** yearly additions grow 15% a year (a project assumption, S20), with the first year solved so the total additions still match the IEA capacity. The base-year build used for the secondary figure is the 2023 to 2024 addition (14 GW; IEA numbers are rounded).

In [ ]:
t = dcp[(dcp.year == 2030) & (dcp.basis == "Whole data centre")].copy()
t = t.pivot(index="iea_case", columns="path", values="build_in_target_year_gw").round(1)
t.columns = ["constant growth: build in 2030 (GW)", "even build: GW a year, 2025-2030"]
t["added 2024-2030 (GW)"] = dcp[(dcp.year == 2030) & (dcp.basis == "Whole data centre") & (dcp.path == "even")].set_index("iea_case")["additions_total_gw"]
t["base-year build 2023-24 (GW)"] = dcp[(dcp.year == 2030) & (dcp.basis == "Whole data centre") & (dcp.path == "even")].set_index("iea_case")["base_build_2023_24_gw"]
t[["added 2024-2030 (GW)", "even build: GW a year, 2025-2030", "constant growth: build in 2030 (GW)", "base-year build 2023-24 (GW)"]]

### 3.3 Main data-centre figure: copper in the capacity built in 2030
Thousand tonnes of copper a year, using 30-40 tonnes per MW (S13, a secondary, low-reliability source). This gross figure is always positive. **It is unresolved whether the 30-40 t/MW refer to total or IT capacity** (known issue K01; S14 did not settle it either), so both are shown side by side: total capacity (main) and IT capacity (second result).

In [ ]:
def dc_table(component_id):
    d = head[(head.component_id == component_id) & (head.year == 2030)].copy()
    d["basis"] = d["basis"].replace({"Whole data centre": "total capacity (main)", "IT equipment": "IT capacity"})
    d["build path"] = d["path_method"].replace({"even": "even", "constant_growth": "constant growth"})
    out = pd.DataFrame({"IEA case": d["variant"], "basis": d["basis"], "build path": d["build path"], "low (kt)": kt(d["low_t"]), "mid (kt)": kt(d["mid_t"]),
                        "high (kt)": kt(d["high_t"]), "mid (% of 2025e mine output)": d["mid_pct_of_mine"].round(1)})
    return out.sort_values(["IEA case", "build path", "basis"]).set_index(["IEA case", "build path", "basis"])


print("Copper in the capacity built in 2030 (main figure; always positive)")
dc_table("dc_new_capacity")

In [ ]:
pair = dcc[(dcc.iea_case == "Base") & (dcc.year == 2030) & (dcc.path == "even") & (dcc.intensity_case == "mid") & (dcc.base_build_case == "2023-24")].set_index("basis")
tot_c, it_c = pair.loc["Whole data centre"], pair.loc["IT equipment"]
print(f"Base Case, 2030, even build, middle copper per MW: IT capacity gives {(1 - it_c.new_capacity_copper_t / tot_c.new_capacity_copper_t) * 100:.0f}% less copper in new capacity "
      f"({it_c.new_capacity_copper_t / 1000:,.0f} against {tot_c.new_capacity_copper_t / 1000:,.0f} kt) and {(1 - it_c.extra_vs_base_t / tot_c.extra_vs_base_t) * 100:.0f}% less extra over the 2024 build "
      f"({it_c.extra_vs_base_t / 1000:,.0f} against {tot_c.extra_vs_base_t / 1000:,.0f} kt).")
c0 = cap[(cap.iea_case == "Base") & (cap.year == 2030)].set_index("basis")["capacity_gw"]
print(f"IT capacity is {(1 - c0['IT equipment'] / c0['Whole data centre']) * 100:.0f}% below total capacity in the Base Case in 2030 ({c0['IT equipment']:.0f} against {c0['Whole data centre']:.0f} GW), "
      "but the difference in the additions since 2024 is smaller, which is why the copper difference is smaller.")

### 3.4 Secondary figure: the extra over the 2024 build
This subtracts the copper that 2024's build (14 GW) used, so it measures how much the *pace* of building changes. **Why some numbers are negative.** In the Headwinds case the IEA capacity rises by 61 GW between 2024 and 2030, an even build of about 10.2 GW a year, which is *less* than the 14 GW built in 2024. Under that path less copper goes into new capacity each year than in 2024. **That is a slower build, not negative demand**: the gross figure in 3.3 (356 kt for Headwinds) stays positive. The constant-growth path ends 2030 at about 14 GW for Headwinds, roughly the 2024 level, so its extra is close to zero.

In [ ]:
print("Extra over the 2024 build (negative = slower build than in 2024, not negative demand)")
dc_table("dc_extra_vs_base_year_build")

### 3.5 An outside figure that does not match
S13 (a secondary article citing S&P Global) reports 1.1 million tonnes of data-centre copper demand for 2025. The model's base-year build (2023 to 2024, 14 GW at 30-40 t/MW) gives 0.42-0.56 million tonnes. S13 is about 2.0 to 2.6 times higher. **This is not reconciled** (known issue K02): the definitions may differ (for example installed stock rather than a year's build, other equipment, or a different year), and the original S&P Global report (S14) has not been read. It is shown, not used.

In [ ]:
r = ctx.loc[["s13_dc_copper_demand_2025_t", "model_dc_build_2024_copper_low_t", "model_dc_build_2024_copper_high_t"], ["value", "source_id", "note"]].copy()
r["value (kt)"] = (r["value"] / 1000).round(0)
r[["value (kt)", "source_id", "note"]]

## 4. The two effects as separate bars, next to today's mine output
Each bar is *equivalent to X% of 2025 world mine production (an estimate, 23 million tonnes)*. Dots are the middle of the source ranges and lines the low to high range. The **EV transition** (against 2025) and the **data-centre** results (against no baseline for the main figure, against the 2024 build for the secondary one) are separate bars. The bottom bar is **the two effects together, on their own baselines**: EV vs 2025 plus data centres vs the 2024 build. Hatched = 2030 EV values are interpolated.
This is not a comparison of demand with future supply. It leaves out recycling, stocks, new mines and price response.

In [ ]:
def pick(year, comp, var, pm, basis):
    d = head[(head.year == year) & (head.component_id == comp) & (head.variant == var) & (head.path_method == pm)]
    if basis:
        d = d[d.basis == basis]
    return d.iloc[0]


fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True)
for ax, year, ev_pm in zip(axes, (2030, 2035), ("linear", "direct")):
    ev_r = pick(year, "ev_transition", "CPS", ev_pm, None)
    dg = pick(year, "dc_new_capacity", "Base", "even", "Whole data centre")
    dx = pick(year, "dc_extra_vs_base_year_build", "Base", "even", "Whole data centre")
    tot_ = {"low": ev_r.low_pct_of_mine + dx.low_pct_of_mine, "mid": ev_r.mid_pct_of_mine + dx.mid_pct_of_mine, "high": ev_r.high_pct_of_mine + dx.high_pct_of_mine}
    spec = [
        ("EV transition\n(against 2025)", ev_r.low_pct_of_mine, ev_r.mid_pct_of_mine, ev_r.high_pct_of_mine, BLUE, 1.0, "///" if year == 2030 else None),
        ("Data centres: copper in new capacity\n(main figure)", dg.low_pct_of_mine, dg.mid_pct_of_mine, dg.high_pct_of_mine, AQUA, 1.0, None),
        ("Data centres: extra over the 2024 build\n(secondary figure)", dx.low_pct_of_mine, dx.mid_pct_of_mine, dx.high_pct_of_mine, AQUA, 0.45, None),
        ("The two effects together,\non their own baselines\n(EV vs 2025 + data centres vs 2024 build)", tot_["low"], tot_["mid"], tot_["high"], None, 1.0, None),
    ]
    y = np.arange(len(spec))[::-1]
    for yi, (label, lo, mid, hi, col, alpha, hatch) in zip(y, spec):
        if col is None:
            ax.barh(yi, mid, height=0.55, facecolor="none", edgecolor=INK, linewidth=1.8)
        else:
            ax.barh(yi, mid, height=0.55, color=col, alpha=alpha, hatch=hatch, edgecolor="white" if hatch else col, linewidth=0 if not hatch else 1.0)
        ax.errorbar(mid, yi, xerr=[[mid - lo], [hi - mid]], fmt="none", ecolor=INK, elinewidth=1.4, capsize=3)
        ax.annotate(f"{mid:.1f}%", (max(hi, mid), yi), xytext=(7, 0), textcoords="offset points", fontsize=9, va="center")
    ax.set_yticks(y)
    ax.set_yticklabels([sp[0].replace("\\n", "\n") for sp in spec], fontsize=8.5)
    ax.set_title(f"{year}" + (" (main year, EV values interpolated)" if year == 2030 else " (IEA scenario year; data-centre values exploratory)"), fontsize=10.5)
    ax.set_xlabel("Copper per year, % of 2025e world mine production")
    ax.axvline(0, color="#8a8984", lw=0.9)
fig.suptitle("How big are the two effects, side by side and together, next to today's mine output?", x=0.01, y=0.995, ha="left", fontweight="bold", fontsize=12)
fig.text(0.01, 0.915, "Reference case (a convenience choice, not the most likely case): CPS, Base Case data centres, even build, total capacity, middle copper figures. Lines = source ranges.", fontsize=8.5, color="#52514e")
fig.subplots_adjust(top=0.80, bottom=0.14, wspace=0.75, left=0.2, right=0.97)
plt.show()

### The other scenarios and the detail
All IEA scenarios, build paths and both capacity bases. Hollow = interpolated 2030 EV values; diamonds = the secondary data-centre figure (extra over the 2024 build, negative when the build slows); circles otherwise. Colour = component.

In [ ]:
rows2030 = [
    ("ev_transition", "CPS", "linear", None, "EV transition, CPS (straight line)"),
    ("ev_transition", "CPS", "constant_growth", None, "EV transition, CPS (constant growth)"),
    ("ev_transition", "STEPS", "linear", None, "EV transition, STEPS (straight line)"),
    ("ev_transition", "STEPS", "constant_growth", None, "EV transition, STEPS (constant growth)"),
    ("car_market_growth", "CPS", "linear", None, "Car-market growth, CPS (derived)"),
    ("dc_new_capacity", "Base", "even", "Whole data centre", "Data centres, new capacity: Base, even, total"),
    ("dc_new_capacity", "Base", "even", "IT equipment", "Data centres, new capacity: Base, even, IT"),
    ("dc_new_capacity", "Base", "constant_growth", "Whole data centre", "Data centres, new capacity: Base, constant growth"),
    ("dc_new_capacity", "Lift-Off", "even", "Whole data centre", "Data centres, new capacity: Lift-Off, even"),
    ("dc_new_capacity", "Headwinds", "even", "Whole data centre", "Data centres, new capacity: Headwinds, even"),
    ("dc_extra_vs_base_year_build", "Base", "even", "Whole data centre", "Secondary, extra over 2024 build: Base, even"),
    ("dc_extra_vs_base_year_build", "Headwinds", "even", "Whole data centre", "Secondary, extra over 2024 build: Headwinds, even*"),
]
rows2035 = [
    ("ev_transition", "CPS", "direct", None, "EV transition, CPS (IEA figure)"),
    ("ev_transition", "STEPS", "direct", None, "EV transition, STEPS (IEA figure)"),
    ("car_market_growth", "CPS", "direct", None, "Car-market growth, CPS (derived)"),
    ("dc_new_capacity", "Base", "even", "Whole data centre", "Data centres, new capacity: Base, even, total"),
    ("dc_new_capacity", "Base", "even", "IT equipment", "Data centres, new capacity: Base, even, IT"),
    ("dc_new_capacity", "Base", "constant_growth", "Whole data centre", "Data centres, new capacity: Base, constant growth"),
    ("dc_new_capacity", "Lift-Off", "even", "Whole data centre", "Data centres, new capacity: Lift-Off, even"),
    ("dc_new_capacity", "Headwinds", "even", "Whole data centre", "Data centres, new capacity: Headwinds, even"),
    ("dc_extra_vs_base_year_build", "Base", "even", "Whole data centre", "Secondary, extra over 2024 build: Base, even"),
    ("dc_extra_vs_base_year_build", "Headwinds", "even", "Whole data centre", "Secondary, extra over 2024 build: Headwinds, even*"),
]
ccol = {"ev_transition": BLUE, "car_market_growth": ORANGE, "dc_new_capacity": AQUA, "dc_extra_vs_base_year_build": AQUA}

fig, axes = plt.subplots(1, 2, figsize=(12, 6.2), sharex=True)
for ax, year, spec in zip(axes, (2030, 2035), (rows2030, rows2035)):
    y = np.arange(len(spec))[::-1]
    for yi, (comp, var, pm, basis, label) in zip(y, spec):
        r_ = pick(year, comp, var, pm, basis)
        c = ccol[comp]
        hollow = r_.is_interpolated == 1
        marker = "D" if comp == "dc_extra_vs_base_year_build" else "o"
        ax.errorbar(r_.mid_pct_of_mine, yi, xerr=[[r_.mid_pct_of_mine - r_.low_pct_of_mine], [r_.high_pct_of_mine - r_.mid_pct_of_mine]],
                    fmt=marker, color=c, ecolor=c, elinewidth=1.5, ms=6.5, mfc="white" if hollow else c, mew=1.7, capsize=0)
        ax.annotate(f"{r_.mid_pct_of_mine:.1f}%", (max(r_.high_pct_of_mine, r_.mid_pct_of_mine), yi), xytext=(6, 0), textcoords="offset points", fontsize=8, va="center", color="#52514e")
    ax.axvline(0, color="#8a8984", lw=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels([sp[4] for sp in spec], fontsize=8)
    ax.set_title(f"{year}" + (" (main year)" if year == 2030 else " (IEA scenario year)"))
    ax.set_xlabel("Copper per year, % of 2025e world mine production")
fig.legend(handles=[Line2D([], [], marker="o", color=BLUE, lw=0, label="EV transition"), Line2D([], [], marker="o", color=ORANGE, lw=0, label="Car-market growth (not an EV effect)"),
                    Line2D([], [], marker="o", color=AQUA, lw=0, label="Data centres, copper in new capacity (main)"),
                    Line2D([], [], marker="D", color=AQUA, lw=0, label="Data centres, extra over 2024 build (secondary)"),
                    Line2D([], [], marker="o", color="#52514e", mfc="white", mew=1.6, lw=0, label="hollow = interpolated")], loc="lower center", ncol=3, fontsize=8)
fig.suptitle("The same bars for every scenario and path", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.text(0.01, 0.945, "*Headwinds with an even build is below the 2024 build pace, so its secondary figure is negative: a slower build, not negative demand.", fontsize=8.5, color="#52514e")
fig.subplots_adjust(top=0.88, bottom=0.18, wspace=0.75, left=0.2, right=0.97)
plt.show()

## 5. Which assumptions move the 2030 result most?
Each bar changes one assumption at a time from the reference case, which is a convenience choice, not the most likely case. The two effects get their own bars first: the **EV transition term alone** and the **data-centre main figure (copper in new capacity) alone**. The last chart is **the two effects together, on their own baselines** (EV transition against 2025 plus the data-centre extra over the 2024 build; the car-market growth term is left out because it is not an EV effect).

In [ ]:
ref = sens[sens.bar_order == 0].iloc[0]


def tornado_frame(col, bar_orders):
    a_ = sens[sens.bar_order.isin(bar_orders)].copy()
    a_["chg"] = (a_[col] - ref[col]) / 1000.0
    rows_ = []
    for bo, g_ in a_.groupby("bar_order"):
        lo_row, hi_row = g_.loc[g_.chg.idxmin()], g_.loc[g_.chg.idxmax()]
        rows_.append({"bar": g_.bar.iloc[0], "low": min(0, lo_row.chg), "high": max(0, hi_row.chg),
                      "low_label": lo_row.alt_label if lo_row.chg < 0 else "", "high_label": hi_row.alt_label if hi_row.chg > 0 else ""})
    out = pd.DataFrame(rows_)
    out["swing"] = out.high - out.low
    return out.sort_values("swing")


def draw_tornado(ax, tb_, title, ref_text):
    for i, r_ in enumerate(tb_.itertuples()):
        if r_.low < 0:
            ax.barh(i, r_.low, left=0, color=BLUE, height=0.55)
            ax.annotate(r_.low_label, (r_.low, i), xytext=(-5, 0), textcoords="offset points", ha="right", va="center", fontsize=8)
        if r_.high > 0:
            ax.barh(i, r_.high, left=0, color=ORANGE, height=0.55)
            ax.annotate(r_.high_label, (r_.high, i), xytext=(5, 0), textcoords="offset points", ha="left", va="center", fontsize=8)
    ax.axvline(0, color=INK, lw=1)
    ax.set_yticks(range(len(tb_)))
    ax.set_yticklabels([textwrap.fill(b_, 34) for b_ in tb_.bar], fontsize=8)
    lim = max(abs(tb_.low.min()), tb_.high.max()) * 2.1
    ax.set_xlim(-lim, lim)
    ax.set_title(title, fontsize=10.5)
    ax.set_xlabel("Change against the reference case (thousand tonnes of copper a year)")
    ax.text(0.0, -0.2, textwrap.fill(ref_text, 95), transform=ax.transAxes, fontsize=8, color="#52514e", va="top")


ev_tb = tornado_frame("ev_transition_t", [1, 2, 3, 4, 5, 6, 7])
dc_tb = tornado_frame("dc_new_capacity_t", [8, 9, 10, 11])
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
draw_tornado(axes[0], ev_tb, "EV transition alone (against 2025)", f"Reference: {ref.ev_transition_t / 1000:,.0f} kt a year ({ref.ev_transition_t / MINE * 100:.1f}% of 2025e mine output).")
draw_tornado(axes[1], dc_tb, "Data centres alone: copper in new capacity (main figure)", f"Reference: {ref.dc_new_capacity_t / 1000:,.0f} kt a year ({ref.dc_new_capacity_t / MINE * 100:.1f}% of 2025e mine output).")
fig.suptitle("Which assumptions move each effect most in 2030?", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.subplots_adjust(left=0.2, right=0.985, bottom=0.24, top=0.86, wspace=0.75)
plt.show()

In [ ]:
tb = tornado_frame("headline_total_t", [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
fig, ax = plt.subplots(figsize=(10, 6.2))
draw_tornado(ax, tb, "Which assumptions move the two effects together most in 2030?",
             f"The two effects together, on their own baselines (EV vs 2025, data centres vs the 2024 build): {ref.headline_total_t / 1000:,.0f} kt a year, equivalent to {ref.headline_total_pct_of_mine:.1f}% of 2025e mine output. "
             "Headwinds: a slower build than 2024, so a negative data-centre extra, not negative demand.")
fig.subplots_adjust(left=0.36, right=0.97, bottom=0.24)
plt.show()

In [ ]:
st = sens.copy()
for c in ["ev_transition_t", "market_growth_t", "dc_extra_vs_base_t", "dc_new_capacity_t", "headline_total_t", "change_vs_reference_t"]:
    st[c] = kt(st[c])
st["headline_total_pct_of_mine"] = st["headline_total_pct_of_mine"].round(2)
st = st.rename(columns={"ev_transition_t": "EV transition (kt)", "market_growth_t": "car-market growth (kt)", "dc_extra_vs_base_t": "data-centre extra over 2024 build (kt, secondary)",
                        "dc_new_capacity_t": "data-centre new capacity (kt, main)", "headline_total_t": "the two effects together (kt)",
                        "headline_total_pct_of_mine": "together, % of 2025e mine output", "change_vs_reference_t": "change vs reference (kt)"})
st[["bar", "alt_label", "EV transition (kt)", "data-centre new capacity (kt, main)", "data-centre extra over 2024 build (kt, secondary)", "car-market growth (kt)",
    "the two effects together (kt)", "together, % of 2025e mine output", "change vs reference (kt)"]].set_index(["bar", "alt_label"])

## 6. Summary of the numbers
Printed from the tables above, so they always match the code.

In [ ]:
def g(comp, var, pm, basis=None, year=2030):
    return pick(year, comp, var, pm, basis)


f1 = lambda v: f"{v / 1000:,.0f}"
for year in (2030, 2035):
    print(f"--- {year} ---")
    for var in ("CPS", "STEPS"):
        for pm in (("linear", "constant_growth") if year == 2030 else ("direct",)):
            r_ = g("ev_transition", var, pm, year=year)
            m_ = g("car_market_growth", var, pm, year=year)
            print(f"EV transition {var} {pm}: {f1(r_.mid_t)} kt ({r_.mid_pct_of_mine:.1f}%), range {f1(r_.low_t)}-{f1(r_.high_t)} kt ({r_.low_pct_of_mine:.1f}-{r_.high_pct_of_mine:.1f}%); "
                  f"car-market growth {f1(m_.mid_t)} kt ({m_.mid_pct_of_mine:.1f}%), band {f1(m_.low_t)}-{f1(m_.high_t)} kt ({m_.low_pct_of_mine:.1f}-{m_.high_pct_of_mine:.1f}%)")
    for case in ("Base", "Lift-Off", "High Efficiency", "Headwinds"):
        for pm in ("even", "constant_growth"):
            for basis in ("Whole data centre", "IT equipment"):
                r_ = g("dc_extra_vs_base_year_build", case, pm, basis, year)
                n_ = g("dc_new_capacity", case, pm, basis, year)
                print(f"DC {case} {pm} {basis}: extra {f1(r_.mid_t)} kt ({r_.mid_pct_of_mine:.1f}%); gross new capacity {f1(n_.mid_t)} kt ({n_.mid_pct_of_mine:.1f}%)")
print("--- sensitivity (2030) ---")
print(f"Reference: EV transition {ref.ev_transition_t / 1000:,.0f} kt ({ref.ev_transition_t / MINE * 100:.1f}%); data-centre copper in new capacity {ref.dc_new_capacity_t / 1000:,.0f} kt ({ref.dc_new_capacity_t / MINE * 100:.1f}%); "
      f"data-centre extra over the 2024 build {ref.dc_extra_vs_base_t / 1000:,.0f} kt ({ref.dc_extra_vs_base_t / MINE * 100:.1f}%); car-market growth {ref.market_growth_t / 1000:,.0f} kt")
print(f"The two effects together, on their own baselines (EV vs 2025 + data centres vs the 2024 build): {ref.headline_total_t / 1000:,.0f} kt = {ref.headline_total_pct_of_mine:.2f}% of 2025e mine output")
for name_, t_ in (("EV transition alone", ev_tb), ("data centres, new capacity alone", dc_tb), ("the two effects together", tb)):
    print(f"Biggest swings, {name_}:")
    for r_ in t_.sort_values("swing", ascending=False).head(6).itertuples():
        print(f"  {r_.bar}: {r_.low:,.0f} to +{r_.high:,.0f} kt")
print(f"Headline total across all alternatives: {sens.headline_total_t.min() / 1000:,.0f} kt ({sens.headline_total_pct_of_mine.min():.1f}%) to {sens.headline_total_t.max() / 1000:,.0f} kt ({sens.headline_total_pct_of_mine.max():.1f}%)")
hy = sens[sens.bar_order == 7].iloc[0]
print(f"Hybrid bounding case: headline {hy.headline_total_t / 1000:,.0f} kt ({hy.headline_total_pct_of_mine:.1f}%), EV transition {hy.ev_transition_t / 1000:,.0f} kt, car-market growth {hy.market_growth_t / 1000:,.0f} kt")
rb = sens[sens.bar_order == 13]
print(f"Rounding of IEA EV sales and shares: car-market growth {rb.market_growth_t.min() / 1000:,.0f} to {rb.market_growth_t.max() / 1000:,.0f} kt")

## 7. What this does and does not show
*(written after reading the results above)*

**What the arithmetic shows (a scenario, not a forecast)**

- **The two effects, side by side, in 2030 (reference case).** The EV transition term is 927 kt of copper a year, equivalent to 4.0% of 2025 world mine output. The main data-centre figure, the copper in the capacity built in 2030, is 752 kt, equivalent to 3.3% of today's mine output. The reference case is a convenience choice, not the most likely case.
- **The two effects together, on their own baselines.** The EV transition term (against 2025) plus the data-centre extra over the 2024 build (262 kt, 1.1%) is 1,189 kt, equivalent to 5.2% of today's mine output. This total adds two figures with different baselines (2025 for cars, 2024 for data centres), which is why it is labelled that way and not shown as one effect.
- **The EV term, and how little the copper figures matter.** Using only the source ranges for copper per car (BEV 82-91 kg, PHEV 59-68 kg, petrol/diesel car 18-22 kg), the 2030 EV transition term runs from 836 to 1,017 kt (3.6 to 4.4%). Much more depends on the path and the IEA scenario: with constant yearly growth to 2030 it is 677 kt (2.9%); in STEPS it is 1,101 kt (4.8%) on a straight line and 792 kt (3.4%) with constant growth.
- **2035 beside 2030.** Straight from the IEA's own scenario year, the EV transition term is 1,853 kt (8.1%) in CPS and 2,202 kt (9.6%) in STEPS. The interpolated 2030 values (2.9 to 4.8%) sit between the 2025 actuals and these figures, so the interpolation fixes the size in 2030 but not the direction: the term grows under every path and both scenarios.
- **Car-market growth, kept apart.** World car sales implied by the IEA's EV sales and shares rise from about 84 million in 2025 to about 99 million in 2030 (straight line). Times the copper in a petrol/diesel car this gives 292 kt (1.3%), with a band of 145 to 463 kt (0.6 to 2.0%) from the IEA's rounding alone. It is not an EV effect and it is not added to the EV transition term. The total sales behind it are derived from rounded EV shares plus interpolation, not from EV data.
- **Hybrids.** The IEA data cannot split hybrids out of the non-EV group, so no data-based sensitivity is possible. As an extreme bound, treating every non-EV car as a hybrid (39-45 kg) would cut the total of the two effects from 1,189 to 883 kt (3.8%): the EV transition term falls to 621 kt, while the car-market growth term rises to 613 kt.
- **Data centres: the main figure, copper in new capacity, depends mostly on the IEA case and the build path.** For the Base Case in 2030 it is 752 kt (3.3%) on an even build of 21.5 GW a year and 1,037 kt (4.5%) if additions grow 15% a year (29.6 GW built in 2030). Lift-Off gives 1,213 kt (5.3%) even and 1,673 kt (7.3%) on constant growth, High Efficiency gives 513 kt (2.2%) even, and Headwinds 356 kt (1.5%). It is always positive.
- **The secondary figure, the extra over the 2024 build, and the negative Headwinds case.** Subtracting the copper in 2024's 14 GW build gives 262 kt (1.1%) for the Base Case on an even build, 547 kt (2.4%) on constant growth, 723 kt (3.1%) for Lift-Off and 23 kt (0.1%) for High Efficiency. For Headwinds on an even build it is -134 kt (-0.6%): the IEA's 61 GW of additions between 2024 and 2030 is about 10.2 GW a year, below the 14 GW built in 2024. That is a slower build, not negative demand: the copper in the capacity built in 2030 is still 356 kt. On constant growth the Headwinds extra is about zero.
- **IT capacity as the second result.** For the Base Case in 2030 with an even build, IT capacity gives 618 kt of copper in new capacity against 752 kt on total capacity (18% lower), and an extra over the 2024 build of 233 kt against 262 kt (11% lower). IT capacity itself is 23% below total capacity in that case, but the difference in the additions since 2024 is smaller, so the copper differs less. **Which capacity the 30-40 t/MW refer to is unresolved** (K01); S14 did not settle it either.
- **An outside figure that does not match.** S13 reports 1.1 Mt of data-centre copper demand for 2025, about 2.0 to 2.6 times the 0.42-0.56 Mt the model gives for the 2024 build. This is **not reconciled** (K02) and is shown, not used.
- **What moves the 2030 result most.** For the EV transition term alone: the hybrid bound (-306 kt), the 2030 path (-249 kt), the IEA scenario (+174 kt), the size of the petrol car replaced (-104 to +153 kt) and copper per BEV (+/-63 kt); copper per PHEV hardly matters because PHEV sales are about flat (7.2 to 7.1 million). For the data-centre main figure alone: the IEA case (-397 to +461 kt), the build path (+285 kt on constant growth), the capacity basis (-134 kt for IT) and copper per MW (+/-108 kt on the main figure; only +/-38 kt on the total of the two effects, because the base-year build is netted out there). For the two effects together the total ranges from 792 kt (3.4%, Headwinds) to 1,650 kt (7.2%, Lift-Off) across all the alternatives.
- **2035 data-centre values are exploratory** (IEA). The even-build average falls after 2030 because the IEA's own additions slow (51 GW over 2030-2035 against 129 GW over 2024-2030), which is why the 2035 data-centre figures on an even build (573 kt, 2.5% for new capacity in the Base Case) are below the 2030 ones.

**What this does not show**

- **A forecast, or a statement about supply.** The percentages compare a copper amount with *today's* mine output (a 2025 estimate). There is no projection of mine supply here (new mines, closures), nor recycling, stocks or the copper price response. "Equivalent to X% of today's mine output" is the only claim made.
- **All uses of copper.** Vans, buses, trucks and two- and three-wheelers have no copper figure in the project data and are left out, and fuel-cell cars as well (about 7,000 sold), so the EV term is understated. Other uses of copper are not covered, and S13's own totals include uses (such as defence) that are not additive with this scenario.
- **Measured copper content.** Copper per car (S11) and per MW (S13) come from secondary articles: medium and low reliability. The original S&P Global report (S14) was checked for the capacity basis and did not settle it.
- **A year-by-year build or any replacement.** The yearly data-centre build is unknown (two stylised paths), only new capacity is counted, and the base-year build used for the secondary figure is a single rounded number (an alternative base is shown). The base years differ (2025 for cars, 2024 for data centres).
- **Exact numbers.** IEA values are rounded to two significant figures, derived car sales are approximate, the 2030 EV sales are interpolated, and the ranges are source ranges, not confidence intervals.
- **Substitution or price effects.** Whether higher copper prices change how much copper cars or data centres use is not modelled.

## 8. Save the derived results
For the dashboard. CSV files go to `results/`, and a database rebuild reloads them as `res_` tables.

In [ ]:
import os

os.makedirs("../results", exist_ok=True)


def save(df, name):
    df.to_csv(f"../results/{name}.csv", index=False)
    df.to_sql(name, con, if_exists="replace", index=False)


sw = tb[["bar", "swing"]].rename(columns={"swing": "swing_kt"})
sens_out = sens.merge(sw, on="bar", how="left")
save(head, "res_demand_headline")
save(sens_out, "res_demand_sensitivity")
save(assump, "res_demand_assumptions")
save(ctx.reset_index(), "res_demand_context")
con.commit()
con.close()
print("Saved 4 result tables")